# Controle S2 : reexecution de l'audit, concordance et prevalence

Ce notebook repond aux recommandations R1, R2 et R3 de l'expertise. Il tourne de haut en bas sur un Colab vide, sans rien modifier a la main.

On fait trois choses :

1. On retelecharge PaySim et on verifie l'empreinte SHA256.
2. On relance l'audit et on compare chaque chiffre du rapport a la sortie du code (R1).
3. On sort les taux de fraude jour par jour, par partition et pour les deux populations (R3).

Le protocole gele (R2) est dans `docs/fiche_protocole.md` et `configs/splits.json`.

In [ ]:
!git clone -q https://github.com/kenzaez/pfa06-graph-fraud.git
%cd pfa06-graph-fraud
!pip install -q -r requirements.txt

## 1. Telechargement de PaySim

Il faut le secret `KAGGLE_API_TOKEN` dans Colab (icone cle a gauche).

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

In [ ]:
!kaggle datasets download -d ealaxi/paysim1 -p data
!unzip -o -q data/paysim1.zip -d data
!ls -lh data

## 2. Audit complet

`src/audit.py` recalcule tous les chiffres du chapitre 3 et les ecrit dans `reports/audit_outputs.json`. Les versions des bibliotheques vont dans `reports/environnement.json`.

In [ ]:
import json
import pandas as pd
from src import audit

PATH = "data/PS_20174392719_1491204439457_log.csv"
df, out = audit.main(PATH)
pd.Series(out).to_frame("valeur")

In [ ]:
json.loads(open("reports/environnement.json").read())

## 3. Tableau de concordance (R1)

Chaque chiffre du rapport est dans `configs/report_values.json`. On compare automatiquement avec ce que le code vient de sortir. Si tout est OK, les chiffres du rapport sont prouves par le code.

In [ ]:
from src import concordance

table = concordance.main()
print(table.statut.value_counts())
table

In [ ]:
table[table.statut != "OK"]

## 4. Prevalence par jour et par partition (R3)

Taux de fraude jour par jour, pour tous les types et pour TRANSFER + CASH_OUT. On marque le regime de volume, les jours atypiques et les jours ou le budget d'alertes depasse le nombre de transactions.

In [ ]:
from src import prevalence

jour, part, sens, sans31 = prevalence.main(df)
pd.set_option("display.max_columns", None)
jour

In [ ]:
from IPython.display import Image

Image("reports/figures/prevalence_par_jour.png")

### Partitions de reference, sur les deux populations

La colonne `ap_hasard` donne l'Average Precision d'un modele au hasard sur chaque partition. C'est la valeur a afficher a cote de chaque score.

In [ ]:
part

### Test sans le jour 31 (resultat diagnostique)

In [ ]:
sans31

### Sensibilite du decoupage (diagnostic seulement)

Ces decoupages ne servent qu'a montrer comment les taux bougent. Le decoupage de reference reste 1 a 445, 446 a 594, 595 a 743. Aucun modele n'est evalue dessus.

In [ ]:
sens

## 5. Tests du protocole

Les tests verifient les partitions, le contrat de variables, les choix geles et la concordance.

In [ ]:
!python -m pytest -q tests

## 6. A recuperer pour le prof

Tout est dans `reports/` : `audit_outputs.json`, `environnement.json`, `tables/concordance.csv`, `tables/prevalence_par_jour.csv`, `tables/prevalence_par_partition.csv`, `tables/sensibilite_decoupage.csv`, `tables/test_sans_jour31.csv` et `figures/prevalence_par_jour.png`.

In [ ]:
!zip -q -r controle_s2.zip reports docs configs
from google.colab import files

files.download("controle_s2.zip")